# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [30]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [32]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [33]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [34]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [36]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [37]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [38]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [39]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [42]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [43]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'}]}

In [44]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Zhihu page', 'url': 'https://www.zhihu.com/org/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [48]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano


In [47]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nconvaiinnovations/laya\nUpdated\n3 days ago\n•\n3.84k\nQwen/Qwen-Image-2.1\nUpdated\n6 days ago\n•\n48.4k\n•\n2.39k\

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a vibrant AI community dedicated to building the future of machine learning. It serves as a collaborative platform where researchers, developers, and enthusiasts worldwide connect to create, share, and innovate around machine learning models, datasets, and applications.

The company provides an extensive hub hosting **over 2 million machine learning models** and **500,000+ datasets**, making it the go-to destination for ML collaboration and discovery. Hugging Face empowers users to host unlimited public models, datasets, and applications, fostering openness and accelerating AI development.

---

## Products & Solutions

- **Hugging Face Hub:** A central repository allowing collaboration on machine learning models, datasets, and applications.
- **Models:** Explore and download a vast selection of trending and cutting-edge AI models, including image, text, and video generation models.
- **Datasets:** Access and contribute to a huge variety of datasets updated regularly for diverse AI tasks.
- **Spaces:** Host and interact with AI applications seamlessly through web spaces.
- **HuggingChat:** AI-driven chat solutions powered by open models.
- **Enterprise Solutions:** Tailored offerings such as **Enterprise Support**, **Inference Providers**, **Inference Endpoints**, and **Storage Buckets** for businesses scaling their AI capabilities.
- **Hugging Face PRO:** Advanced tools and features designed for teams and enterprises.

---

## Community & Collaboration

Hugging Face is more than just a platform—it's a thriving global community of AI researchers, developers, and enthusiasts. The community is active across multiple channels including:

- Discord
- Forum
- GitHub
- Blog and Daily Papers posts

This openness drives innovation, knowledge sharing, and real-world applications of AI technologies.

---

## Company Culture

- **Innovative:** Constantly pushing the boundaries of machine learning through collaboration and open-source.
- **Open & Inclusive:** Encouraging public sharing of models and datasets that anyone can access and improve.
- **Community-Focused:** Believes in the power of a connected AI community to build the future together.
- **Supportive:** Provides enterprise-grade support to help organizations leverage AI responsibly and efficiently.

---

## Careers at Hugging Face

Join a mission-driven company at the heart of cutting-edge AI development. Hugging Face seeks talented individuals passionate about machine learning, open-source collaboration, and community building. Career opportunities span engineering, research, product management, enterprise solutions, and more.

Working at Hugging Face means being part of a fast-paced, innovative environment that values creativity, teamwork, and impact.

Explore open positions and join the AI revolution!

---

## Who Uses Hugging Face?

- AI researchers and developers worldwide
- Startups and SMEs deploying AI-powered apps
- Large enterprises scaling machine learning solutions
- Educators and students accessing open resources
- Innovators building next-generation AI products

---

## Connect and Learn More

- Visit: [huggingface.co](https://huggingface.co)
- Join the Community Forums and Discord
- Explore GitHub repositories and open-source projects
- Follow their blog for latest AI news and papers

---

**Hugging Face: The AI community building the future, together.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
from IPython.display import Markdown, display, update_display

if "openai" not in globals():
    from openai import OpenAI
    openai = OpenAI()

if "brochure_system_prompt" not in globals():
    brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)

In [28]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the premier **collaboration platform for the machine learning community**, committed to building the future of AI through open, ethical, and community-driven innovation. Serving as the **home of machine learning**, Hugging Face empowers a fast-growing global community of ML engineers, scientists, and end users to **create, discover, share, and experiment with models, datasets, and AI applications**.

With over **2 million models** and **500,000 datasets** hosted on its Hub, Hugging Face acts as a central hub for open source ML resources to accelerate research and deployment. Their mission is to **build an open and ethical AI future together**, leveraging cutting-edge technology and collective intelligence.

---

## Platform & Solutions

- **Models & Datasets:** Easy access and collaboration on millions of open-source models and datasets.
- **Spaces:** Host and deploy machine learning apps and demos effortlessly.
- **Inference Endpoints & Providers:** Enterprise-grade scalable ML model inference for real-world applications.
- **Storage Buckets:** Secure data storage tailored for ML workloads.
- **HuggingChat:** Featuring AI-driven chat and interaction tools.
- **Enterprise Services:** Custom team & enterprise plans with dedicated support and professional services.
  
Hugging Face offers robust APIs and tools for **model training, evaluation, hosting, and deployment**, enabling organizations to build AI solutions faster and smarter.

---

## Community & Culture

Hugging Face thrives on an **inclusive, open culture** that encourages collaboration, knowledge sharing, and transparency. Its global community participates actively through:

- Forums, Discord, and GitHub discussions
- Public contributions to models and datasets
- Daily AI and ML research papers and blog updates
- Educational resources and tutorials to nurture ML skills

The company fosters an environment where machine learning professionals from diverse backgrounds can innovate **openly and ethically**, aligning on a shared vision of AI for good.

---

## Customers & Collaborators

Hugging Face serves a broad spectrum of users—from researchers and individual developers to large enterprises—across industries including:

- Technology & SaaS companies
- Academia and research institutions
- Healthcare and biotech
- Automotive and robotics
- Finance and media

Their enterprise solutions are tailored to support teams requiring **scalable AI infrastructure, dedicated support, and collaborative workflows**.

---

## Careers at Hugging Face

Join a **talented science and engineering team** exploring the edge of AI technology! Hugging Face offers rewarding career opportunities for those passionate about advancing machine learning and AI ethics.

Work at Hugging Face if you want to:

- Build tools and libraries used by millions worldwide
- Engage with a vibrant, mission-driven community
- Collaborate on groundbreaking open source projects
- Contribute to shaping the future of AI technology and policy

Visit Hugging Face’s **Careers** page to explore open roles ranging from research scientists and engineers to product and community managers.

---

## Key Values

- **Open Source & Collaboration:** Promote a transparent, freely accessible AI ecosystem.
- **Ethical AI:** Commit to building responsible AI with fairness and privacy in mind.
- **Innovation at Scale:** Deliver tools and infrastructure that scale from hobbyists to enterprises.
- **Community Empowerment:** Enable learning and participation across skill levels and backgrounds.

---

## Connect With Hugging Face

- **Website:** https://huggingface.co  
- **Community Forum & Discord:** Engage with fellow ML enthusiasts and experts  
- **GitHub:** Access and contribute to open source repositories  
- **Social:** Twitter, LinkedIn - follow for updates and insights  

---

Hugging Face is **building the future of AI — together.** Whether you’re an individual innovator, part of an enterprise, or looking to build your career in AI, Hugging Face offers the platform, community, and tools to bring your ideas to life.

In [29]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a pioneering AI company and vibrant collaboration platform where the global machine learning (ML) community comes together to build the future of AI. It provides an open and inclusive space to create, discover, and share machine learning models, datasets, and applications—enabling researchers, developers, and organizations to innovate faster and more efficiently.

With over **2 million models**, **500,000 datasets**, and **1 million+ applications**, Hugging Face has become the central hub for AI and ML projects, fostering unprecedented collaboration and accelerating the pace of technological advancement.

---

## What Hugging Face Offers

### Models  
Browse and collaborate on an extensive library of ML models for a wide range of AI tasks. These models are updated frequently and cover domains such as natural language processing, computer vision, and multimodal AI.

### Datasets  
Access hundreds of thousands of datasets curated for AI training and benchmarking. These resources span multiple industries, enabling highly specialized and diverse research.

### Spaces  
Spaces are interactive ML-powered demos and applications created by the community, enabling users to experiment with cutting-edge AI directly in their browsers.

### Buckets & Storage  
Manage and store large amounts of data efficiently with Hugging Face's cloud storage infrastructure designed for scalable ML workflows.

### Enterprise Solutions  
Tailored services including enterprise support, inference endpoints, and dedicated infrastructure to help businesses integrate advanced AI into their operations with reliability and performance.

---

## Community & Culture

Hugging Face thrives as a global community where developers, researchers, and enterprises unite to share knowledge and capabilities. Their culture embodies:

- **Open Collaboration:** Encouraging transparency and teamwork by hosting unlimited public projects and fostering knowledge exchange through forums, Github, Discord, and more.
- **Innovation-Driven:** Constantly pushing AI boundaries by updating trending models and datasets.
- **Inclusivity:** Welcoming contributors from all backgrounds to build AI that benefits everyone.
- **Learning & Support:** Providing extensive documentation, tutorials, blogs, and daily paper highlights to empower users at all skill levels.

---

## Customers & Partners

Hugging Face's platform is trusted by thousands of organizations—from research labs and startups to global enterprises—looking to leverage state-of-the-art AI capabilities. Their enterprise offerings allow seamless integration of AI models at scale, backed with dedicated support and optimized cloud infrastructure.

---

## Careers at Hugging Face

Join a passionate, innovative team focused on shaping the future of AI through open collaboration and cutting-edge research. Hugging Face values:

- Creativity and technical excellence
- Collaborative mindset and open-source spirit
- A drive to impact real-world AI applications positively

Open roles span software engineering, research, community management, and enterprise solutions. The company offers a growth-oriented environment where your contributions help build tools that empower millions worldwide.

Explore careers and contribute to the AI revolution with Hugging Face.

---

## Brand & Identity

Hugging Face is recognized for its friendly, accessible brand, with a distinctive color palette centered on vibrant yellows and energetic oranges (#FFD21E, #FF9D00), symbolizing optimism and creativity.

---

## Connect & Learn More

- Visit: [Hugging Face Website](https://huggingface.co)  
- Join the Community: Discord, Forum, GitHub  
- Explore: Models, Datasets, Spaces, and AI Applications

---

**Hugging Face: Empowering the machine learning community to build smarter, faster, and more inclusive AI.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>